# Theo dõi đặc trưng UFM-Rec
Notebook chỉ đọc trạng thái; **Run All không khởi động job GPU**.
Job Foundation Model chờ lượt TF-IDF Transformer kết thúc và GPU rảnh, sau đó chạy smoke 128 sản phẩm rồi trích toàn catalog nếu smoke đạt.
Trích đặc trưng không phải huấn luyện UFM hoàn chỉnh; test chưa được mở.

In [ ]:
from pathlib import Path
import json, subprocess
ROOT = Path('/home/coder/iDragonCloud/DA_AI')
DATA = ROOT / 'data/processed/toys_games_full_temporal'
status = ROOT / 'runs/foundation_queue_v1.json'
print('TRẠNG THÁI JOB:', status.read_text() if status.exists() else 'Chưa có trạng thái queue')
history = ROOT / 'runs/content_transformer_v1/history.jsonl'
if history.exists():
    epochs = [json.loads(x) for x in history.read_text().splitlines()]
    print('EPOCH ĐỐI CHỨNG ĐÃ XONG:', len(epochs))
    if epochs:
        best = max(epochs, key=lambda x: x['validation']['known_user']['ndcg@10'])
        print('BEST VALIDATION:', best['epoch'], best['validation']['known_user'])
print('ĐỐI CHỨNG HOÀN THÀNH:', (ROOT / 'runs/content_transformer_v1/completed.json').exists())
for name, folder in [('SMOKE 128', ROOT / 'runs/foundation_smoke_128_v1'), ('FULL CATALOG', DATA / 'foundation_clip_b32_v1')]:
    complete, progress = folder / 'complete.json', folder / 'progress.json'
    if complete.exists():
        m = json.loads(complete.read_text())
        print(name, 'HOÀN THÀNH:', m['rows'], '| text:', m['has_text'], '| ảnh thành công:', m['has_image'])
    elif progress.exists():
        m = json.loads(progress.read_text())
        print(name, 'ĐÃ LƯU:', m['next_row'], '/', m['config']['rows'], '| trạng thái ảnh:', m['image_status_counts'])
    else:
        print(name, 'CHƯA BẮT ĐẦU')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu', '--format=csv'], capture_output=True, text=True).stdout)


In [ ]:
for filename in ['foundation_catalog_build_v1.log', 'foundation_queue_v1.log']:
    path = ROOT / 'runs' / filename
    print('LOG:', path)
    if path.exists():
        print(subprocess.run(['tail', '-n', '10', str(path)], capture_output=True, text=True).stdout)
    else:
        print('Chưa có log.')
